In [1]:
import numpy as np
import geopandas as gpd
from pathlib import Path
from tqdm import tqdm
import os

In [2]:
names = locals()

In [3]:
ROOT = Path.cwd().parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"

In [ ]:
# Reproject to Eckert IV (equal-area) so that:
# (1) geometry.area returns m², (2) sjoin runs in a common CRS,
# (3) results align with downstream Eckert IV rasters.
quarries = gpd.read_file(DATA / 'global-quarries-210101.gpkg').to_crs('ESRI:54012')
maus = gpd.read_file(DATA / 'global_mining_polygons_v2.gpkg').to_crs('ESRI:54012')

In [ ]:
# Maus polygons represent non-aggregate mines (metals, coal, industrial minerals).
# Any OSM quarry intersecting them is excluded from the construction-aggregate set.
# Left join keeps every OSM quarry; index_right is non-null where overlap exists.
joined = gpd.sjoin(quarries, maus[['geometry']], how='left', predicate='intersects')

# A quarry intersecting multiple Maus polygons appears in multiple rows;
# .unique() collapses to one ID per quarry.
overlap_ids = joined[joined.index_right.notna()].index.unique()

aggregate_quarries = quarries.drop(index=overlap_ids).copy()

In [15]:
print(f"Original: {len(quarries):,}")
print(f"Overlap with Maus (excluded): {len(overlap_ids):,}")
print(f"Aggregate-only retained: {len(aggregate_quarries):,}")

Original: 148,214
Overlap with Maus (excluded): 11,548
Aggregate-only retained: 136,666


In [ ]:
# Area in m² is valid because the CRS is equal-area.
aggregate_quarries['area_m2'] = aggregate_quarries.geometry.area
print(f"Total aggregate quarry area: {aggregate_quarries['area_m2'].sum()/1e6:.1f} km²")

Total aggregate quarry area: 19141.7 km²


In [22]:
aggregate_quarries.to_file(
    DATA / 'global-aggregate-quarries-210101.gpkg',
    driver='GPKG',
)

count the quarry numbers

In [ ]:
# Folder containing your .gpkg files
osm_folder = 'F:/data/osm/210101/'
regions = ['europe', 'north-america', 'asia', 'africa', 'south-america', 'australia-oceania', 'central-america']
print(f'{"Region":<25}{"Polygon count":>15}')
print('-' * 40)

total = 0
for region in regions:
    fp = os.path.join(osm_folder, f'{region}-210101-quarries.gpkg')
    gdf = gpd.read_file(fp)
    n = len(gdf)
    total += n
    print(f'{region:<25}{n:>15,}')

print('-' * 40)
print(f'{"Total":<25}{total:>15,}')

# Optional: also check the global file as a cross-check
fp_global = os.path.join(osm_folder, 'global-quarries-210101.gpkg')
gdf_global = gpd.read_file(fp_global)
print(f'\nGlobal file count: {len(gdf_global):,}')

Region                     Polygon count
----------------------------------------
europe                            67,409
north-america                     25,981
asia                              25,058
africa                            14,785
south-america                     10,938
australia-oceania                  3,134
central-america                      909
----------------------------------------
Total                            148,214

Global file count: 148,214


In [22]:
maus = gpd.read_file(DATA / 'global_mining_polygons_v2.gpkg').to_crs('ESRI:54012')

regions_files = {
    'Europe': 'europe-210101-quarries.gpkg',
    'North America': 'north-america-210101-quarries.gpkg',
    'Asia': 'asia-210101-quarries.gpkg',
    'Africa': 'africa-210101-quarries.gpkg',
    'South America': 'south-america-210101-quarries.gpkg',
    'Australia & Oceania': 'australia-oceania-210101-quarries.gpkg',
    'Central America': 'central-america-210101-quarries.gpkg',
}

results = {}
for region, fp in regions_files.items():
    DATA = Path(r'F:/data/osm/210101/')
    q = gpd.read_file(DATA / fp).to_crs('ESRI:54012')
    joined = gpd.sjoin(q, maus[['geometry']], how='left', predicate='intersects')
    overlap_ids = joined[joined.index_right.notna()].index.unique()
    n_filtered = len(q) - len(overlap_ids)
    results[region] = {'raw': len(q), 'filtered': n_filtered}
    print(f'{region:<22}  raw: {len(q):>7,}  filtered: {n_filtered:>7,}')

total_raw = sum(v['raw'] for v in results.values())
total_filt = sum(v['filtered'] for v in results.values())
print(f'\n{"Total":<22}  raw: {total_raw:>7,}  filtered: {total_filt:>7,}')

Europe                  raw:  67,409  filtered:  65,097
North America           raw:  25,981  filtered:  23,992
Asia                    raw:  25,058  filtered:  22,146
Africa                  raw:  14,785  filtered:  13,998
South America           raw:  10,938  filtered:   8,778
Australia & Oceania     raw:   3,134  filtered:   1,809
Central America         raw:     909  filtered:     846

Total                   raw: 148,214  filtered: 136,666
